# Rasch Measurement Model: CML vs Bayesian Analysis

**목차 / Table of Contents**

1. CML vs Bayesian — 개념 비교
2. Infit / Outfit MNSQ — 수학적 정의
3. Bayesian 적합도 지표
4. 실험 데이터 생성 (Simulation)
5. CML 추정 + Infit/Outfit 계산
6. Stan/Bayesian 추정 + Bayesian 적합도
7. Person parameter 비교 (True vs CML vs Bayes)
8. Item parameter 비교 (True vs CML vs Bayes)
9. 소규모 클래스(≤50명)에서의 RMM 적용 가능성
10. CTT 대비 RMM 권장 이유
11. RMM 이 CTT 보다 진정 필요한 경우


In [ ]:
import sys, os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.optimize import minimize
from scipy.stats import norm

# ── Korean font setup ──────────────────────────────────────────────────────
def _setup_korean_font():
    _candidates = {
        'win32':  [('C:/Windows/Fonts/malgun.ttf',  'Malgun Gothic'),
                   ('C:/Windows/Fonts/gulim.ttc',   'Gulim')],
        'darwin': [('/System/Library/Fonts/AppleSDGothicNeo.ttc', 'Apple SD Gothic Neo'),
                   ('/Library/Fonts/NanumGothic.ttf', 'NanumGothic')],
        'linux':  [('/usr/share/fonts-droid-fallback/truetype/DroidSansFallback.ttf', 'Droid Sans Fallback'),
                   ('/usr/share/fonts/truetype/nanum/NanumGothic.ttf', 'NanumGothic')],
    }
    for path, name in _candidates.get(sys.platform, _candidates['linux']):
        if os.path.exists(path):
            fm.fontManager.addfont(path)
            matplotlib.rcParams['font.family'] = ['DejaVu Sans', name]
            return
    matplotlib.rcParams['font.family'] = 'DejaVu Sans'
_setup_korean_font()
matplotlib.rcParams['axes.unicode_minus'] = False

# ── Stan availability ──────────────────────────────────────────────────────
STAN_AVAILABLE = False
try:
    import cmdstanpy
    STAN_AVAILABLE = True
    print('CmdStanPy available')
except ImportError:
    print('CmdStanPy not found -- Bayesian cells will use synthetic posteriors')

np.random.seed(42)
print('Setup complete')

## 1. CML vs Stan/Bayesian — 개념 비교

### Conditional Maximum Likelihood (CML)

Rasch 모델에서 피험자 파라미터 $\\theta_j$는 **충분 통계량** $r_j = \\sum_i X_{ij}$로 요약됩니다.  
CML은 이 점수에 조건부로 likelihood를 구성하여 문항 파라미터 $b_i$만을 추정합니다.

$$\\mathcal{L}_{CML}(\\mathbf{b}) = \\prod_j \\frac{\\exp(-r_j \\bar{b}_{(j)})}{\\gamma_{r_j}(\\mathbf{b})}$$

- $\\gamma_r(\\mathbf{b})$: Elementary Symmetric Polynomial (ESP) — 가능한 모든 $r$-정답 패턴의 합
- $\\theta_j$가 소거되므로 **피험자 분포 가정 불필요** (distribution-free)
- 문항 파라미터는 **표본 불변성** (specific objectivity) 달성

### Stan/Bayesian

Stan은 피험자와 문항 파라미터를 **동시에** 사전분포와 함께 추정합니다.

$$p(\\boldsymbol{\\theta}, \\mathbf{b} \\mid \\mathbf{X}) \\propto p(\\mathbf{X}|\\boldsymbol{\\theta},\\mathbf{b})\\, p(\\boldsymbol{\\theta})\\, p(\\mathbf{b})$$

| 항목 | CML | Bayesian (Stan) |
|------|-----|------------------|
| 피험자 분포 가정 | 불필요 | 필요 (e.g. $\\theta \\sim N(0,1)$) |
| 결과물 | 점 추정치 + SE | 사후 분포 (MCMC 샘플) |
| 소표본 안정성 | 피험자수 작을 때 추정 불안정 | 사전분포가 정규화 역할 |
| 적합도 | Infit/Outfit MNSQ | PPP, 사후 예측 분포 |
| 계산 속도 | 빠름 | 느림 (MCMC) |
| 파라미터 불확실성 | 표준 오차 (asymptotic) | 사후 분포 (exact) |

> **핵심**: CML은 문항 파라미터의 불편 추정을 위한 엄밀한 방법이며, Bayesian은 불확실성을 분포로 표현하고 복잡한 모델 확장에 유리합니다.


## 2. Infit / Outfit MNSQ — 수학적 정의

피험자 $j$, 문항 $i$에 대해:

$$p_{ij} = \\frac{e^{\\theta_j - b_i}}{1 + e^{\\theta_j - b_i}}, \\quad w_{ij} = p_{ij}(1-p_{ij}), \\quad e_{ij} = X_{ij} - p_{ij}$$

### Outfit MNSQ (문항 $i$에 대해)

$$\\text{OUTFIT}_i = \\frac{1}{J}\\sum_{j=1}^J \\frac{e_{ij}^2}{w_{ij}}$$

- **비가중 평균**: 극단 능력 피험자의 잔차에 민감 (outlier-sensitive)
- 해석: 기대값 1. <0.7 = 과적합(overfit), >1.3 = 부적합

### Infit MNSQ (문항 $i$에 대해)

$$\\text{INFIT}_i = \\frac{\\sum_{j=1}^J e_{ij}^2}{\\sum_{j=1}^J w_{ij}}$$

- **정보 가중 평균**: $w_{ij}$가 클수록 (능력≈난이도) 더 많이 반영
- 극단 피험자에 덜 민감 → 문항 타당도의 핵심 지표

### 판정 기준 (Wright & Stone, 1979)

| MNSQ 범위 | 해석 |
|-----------|------|
| < 0.7 | 과도한 규칙성 (데이터가 너무 잘 맞음, 문항 중복 의심) |
| 0.7 – 1.3 | 적합 (acceptable fit) |
| > 1.3 | 부적합 (misfitting, 무관한 차원 또는 추측 패턴) |
| > 2.0 | 심각한 부적합 |

> Infit/Outfit 모두 확인하되, **Infit 우선**. Outfit이 크더라도 능력 분포의 극단값 때문일 수 있음.


## 3. Bayesian 적합도 지표

Stan/Bayesian 분석에서는 MCMC 샘플로부터 다음 지표를 계산합니다.

### (a) Posterior Predictive p-value (PPP)

$$\\text{PPP} = P(T(\\mathbf{X}^{rep}) \\geq T(\\mathbf{X}^{obs}) \\mid \\mathbf{X}^{obs})$$

- $T$: 테스트 통계량 (예: Infit MNSQ, 문항 분산, etc.)
- $\\mathbf{X}^{rep}$: 사후 샘플에서 생성한 복제 데이터
- PPP $\\approx 0.5$: 완벽한 적합. PPP $< 0.05$ 또는 $> 0.95$: 부적합

### (b) 사후 Infit 분포

각 MCMC 샘플 $s$에서 $\\theta^{(s)}, b^{(s)}$를 이용해 Infit을 계산한 뒤 **분포**로 표현:

$$\\text{Infit}_i^{(s)} = \\frac{\\sum_j e_{ij}^{(s)2}}{\\sum_j w_{ij}^{(s)}}$$

- 95% credible interval이 [0.7, 1.3]을 벗어나면 부적합

### (c) WAIC / LOO-CV

- **WAIC** (Watanabe-AIC): 예측 정확도 + 유효 파라미터 수 벌점
- **LOO** (Leave-One-Out CV): 교차검증 근사. loo 패키지로 계산
- 모델 **비교**에 사용 (절대적 적합도 지표 아님)

### 활용 방안

1. **PPP 계산**: 의심 문항 표시 후 내용 전문가 검토 의뢰
2. **사후 Infit CI**: Infit이 1.3 초과이면서 CI 하한도 1.0 이상 → 제거 권고
3. **LOO**: 문항 포함/제외 모델 비교 → 최적 문항 세트 선정
4. **PPP < 0.05**: 해당 문항의 Rasch 단일차원성 위반 가능성 → DIF 분석 추가 실시

> **요약**: Bayesian 적합도는 단순 합/부합 판정이 아닌 *불확실성을 반영한 분포*로 제공되므로, 소표본에서 특히 유리합니다.


In [ ]:
# ── 4. Simulation: 실험 데이터 생성 ──────────────────────────────────────
np.random.seed(2024)

J = 60   # persons
I = 15   # items

# True parameters (sum-to-zero for items)
theta_true = np.random.normal(0, 1, J)
b_raw = np.random.normal(0, 1, I)
b_true = b_raw - b_raw.mean()   # sum-to-zero constraint

# Generate binary response matrix
def rasch_prob(theta, b):
    logit = theta[:, None] - b[None, :]   # (J, I)
    return 1.0 / (1.0 + np.exp(-logit))

P_true = rasch_prob(theta_true, b_true)
X = (np.random.rand(J, I) < P_true).astype(float)

# Remove extreme scorers (all 0 or all 1)
row_scores = X.sum(axis=1)
valid = (row_scores > 0) & (row_scores < I)
X = X[valid]; theta_true = theta_true[valid]; row_scores = row_scores[valid]
J = X.shape[0]

print(f'J={J} persons, I={I} items')
print(f'True b range   : [{b_true.min():.2f}, {b_true.max():.2f}]')
print(f'True theta range: [{theta_true.min():.2f}, {theta_true.max():.2f}]')
print(f'Response matrix shape: {X.shape}')

In [ ]:
# ── 5. CML 추정 + Infit/Outfit ────────────────────────────────────────────
from scipy.optimize import minimize
from functools import lru_cache

def compute_esp(b_vec):
    """Elementary Symmetric Polynomials via sum algorithm (Gustafsson 1980)."""
    I = len(b_vec)
    # gamma[r] = sum of products of exp(-b) for all size-r subsets
    q = np.exp(-b_vec)           # exp(-b_i)
    gamma = np.zeros(I + 1)
    gamma[0] = 1.0
    for i in range(I):
        for r in range(min(i + 1, I), 0, -1):
            gamma[r] += gamma[r - 1] * q[i]
    return gamma

def cml_nll(b_free, X, fixed_idx=0):
    I = X.shape[1]
    b = np.zeros(I)
    mask = np.ones(I, dtype=bool); mask[fixed_idx] = False
    b[mask] = b_free
    # Sum-to-zero: fix b[0] = -sum(rest)
    b[fixed_idx] = -b_free.sum()
    gamma = compute_esp(b)
    scores = X.sum(axis=1).astype(int)
    nll = 0.0
    for j in range(X.shape[0]):
        r = scores[j]
        if r == 0 or r == I: continue
        num = -np.dot(X[j], b)   # log numerator
        if gamma[r] <= 0: continue
        nll += -(num - np.log(gamma[r]))
    return nll

# Optimize
b_init = np.zeros(I - 1)
res = minimize(cml_nll, b_init, args=(X,), method='L-BFGS-B',
               options={'maxiter': 2000, 'ftol': 1e-12})
b_cml = np.zeros(I)
b_cml[1:] = res.x
b_cml[0]  = -res.x.sum()

# Person MLE
def theta_mle(x_j, b):
    r = x_j.sum()
    if r == 0: return -4.0
    if r == len(b): return 4.0
    def nll(th): p = 1/(1+np.exp(-(th-b))); return -np.sum(x_j*np.log(p+1e-15)+(1-x_j)*np.log(1-p+1e-15))
    return minimize(nll, 0.0, method='Brent', bounds=[(-6,6)]).x if False else minimize(nll, [0.0], method='L-BFGS-B', bounds=[(-6,6)]).x[0]

theta_cml = np.array([theta_mle(X[j], b_cml) for j in range(J)])

# Infit / Outfit
P = rasch_prob(theta_cml, b_cml)   # (J, I)
W = P * (1 - P)                    # variance weights
E = X - P                          # residuals

infit_item  = (E**2).sum(axis=0) / W.sum(axis=0)
outfit_item = ((E**2 / (W + 1e-15)).mean(axis=0))
infit_pers  = (E**2).sum(axis=1) / W.sum(axis=1)
outfit_pers = ((E**2 / (W + 1e-15)).mean(axis=1))

print('Item  | b_true | b_cml | Infit | Outfit')
print('-' * 45)
for i in range(I):
    flag = '  <<' if infit_item[i] > 1.3 or infit_item[i] < 0.7 else ''
    print(f'Item {i+1:2d}| {b_true[i]:+.3f} | {b_cml[i]:+.3f} | {infit_item[i]:.3f} | {outfit_item[i]:.3f}{flag}')

In [ ]:
# ── 6. Stan/Bayesian 추정 + Bayesian 적합도 ──────────────────────────────
STAN_CODE = '''
data {
  int<lower=1> J;  // persons
  int<lower=1> I;  // items
  array[J, I] int<lower=0, upper=1> X;
}
parameters {
  vector[J] theta;
  vector[I] b_raw;
}
transformed parameters {
  vector[I] b = b_raw - mean(b_raw);
}
model {
  theta ~ normal(0, 1);
  b_raw ~ normal(0, 1);
  for (j in 1:J)
    for (i in 1:I)
      X[j][i] ~ bernoulli_logit(theta[j] - b[i]);
}
generated quantities {
  array[J, I] int X_rep;
  for (j in 1:J)
    for (i in 1:I)
      X_rep[j][i] = bernoulli_logit_rng(theta[j] - b[i]);
}
'''

if STAN_AVAILABLE:
    import tempfile, os
    with tempfile.NamedTemporaryFile(suffix='.stan', mode='w', delete=False) as fp:
        fp.write(STAN_CODE); stan_path = fp.name
    model = cmdstanpy.CmdStanModel(stan_file=stan_path)
    fit = model.sample(
        data={'J': J, 'I': I, 'X': X.astype(int).tolist()},
        chains=2, iter_sampling=800, iter_warmup=400,
        show_progress=False
    )
    theta_samples = fit.stan_variable('theta')  # (S, J)
    b_samples     = fit.stan_variable('b')       # (S, I)
    X_rep_samples = fit.stan_variable('X_rep')   # (S, J, I)
    theta_bayes   = theta_samples.mean(axis=0)
    theta_bayes_sd= theta_samples.std(axis=0)
    b_bayes       = b_samples.mean(axis=0)
    b_bayes_sd    = b_samples.std(axis=0)
    print('Stan sampling complete')
else:
    # Synthetic posterior: CML estimates + calibrated noise
    S = 800
    b_bayes_sd    = np.full(I, 0.15)
    theta_bayes_sd= np.full(J, 0.22)
    b_samples     = b_cml + np.random.normal(0, 0.15, (S, I))
    theta_samples = theta_cml + np.random.normal(0, 0.22, (S, J))
    b_bayes       = b_samples.mean(axis=0)
    theta_bayes   = theta_samples.mean(axis=0)
    # Generate X_rep
    X_rep_samples = np.zeros((S, J, I), dtype=int)
    for s in range(S):
        P_s = rasch_prob(theta_samples[s], b_samples[s])
        X_rep_samples[s] = (np.random.rand(J, I) < P_s).astype(int)
    print('Synthetic posterior generated (Stan not available)')

# Posterior Predictive p-value for each item (Infit-based)
def compute_infit_per_item(X_mat, theta_vec, b_vec):
    P = rasch_prob(theta_vec, b_vec)
    W = P * (1 - P)
    E = X_mat - P
    return (E**2).sum(axis=0) / W.sum(axis=0)

infit_obs = compute_infit_per_item(X, theta_bayes, b_bayes)
infit_rep_list = []
for s in range(len(b_samples)):
    inf_s = compute_infit_per_item(X_rep_samples[s], theta_samples[s], b_samples[s])
    infit_rep_list.append(inf_s)
infit_rep = np.array(infit_rep_list)  # (S, I)

PPP = (infit_rep >= infit_obs).mean(axis=0)

# Posterior infit distribution per item
infit_posterior = []
for s in range(len(b_samples)):
    inf_s = compute_infit_per_item(X, theta_samples[s], b_samples[s])
    infit_posterior.append(inf_s)
infit_posterior = np.array(infit_posterior)  # (S, I)
infit_bayes_mean = infit_posterior.mean(axis=0)
infit_bayes_lo   = np.percentile(infit_posterior, 2.5, axis=0)
infit_bayes_hi   = np.percentile(infit_posterior, 97.5, axis=0)

print('\nItem  | Infit(CML) | Infit_B mean [95% CI]          | PPP')
print('-' * 62)
for i in range(I):
    flag = ' <<' if infit_bayes_lo[i] > 1.3 or infit_bayes_hi[i] < 0.7 else ''
    print(f'Item {i+1:2d}| {infit_item[i]:.3f}      | {infit_bayes_mean[i]:.3f} [{infit_bayes_lo[i]:.3f}, {infit_bayes_hi[i]:.3f}] | {PPP[i]:.3f}{flag}')

In [ ]:
# ── 7. Person Parameter 비교: True vs CML vs Bayesian ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# (a) True vs CML
ax = axes[0]
ax.scatter(theta_true, theta_cml, alpha=0.6, s=30, color='steelblue')
lim = max(abs(theta_true).max(), abs(theta_cml).max()) + 0.3
ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=1)
ax.set_xlabel('True θ (logit)'); ax.set_ylabel('CML θ (logit)')
ax.set_title('Person θ: True vs CML')
corr_cml = np.corrcoef(theta_true, theta_cml)[0, 1]
ax.text(0.05, 0.92, f'r = {corr_cml:.3f}', transform=ax.transAxes, fontsize=10)

# (b) True vs Bayesian
ax = axes[1]
ax.errorbar(theta_true, theta_bayes, yerr=1.96*theta_bayes_sd,
            fmt='o', alpha=0.5, capsize=2, markersize=4, color='darkorange')
ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=1)
ax.set_xlabel('True θ (logit)'); ax.set_ylabel('Posterior mean θ')
ax.set_title('Person θ: True vs Bayesian (±1.96 SD)')
corr_bay = np.corrcoef(theta_true, theta_bayes)[0, 1]
ax.text(0.05, 0.92, f'r = {corr_bay:.3f}', transform=ax.transAxes, fontsize=10)

# (c) RMSE comparison
ax = axes[2]
rmse_cml = np.sqrt(np.mean((theta_cml - theta_true)**2))
rmse_bay = np.sqrt(np.mean((theta_bayes - theta_true)**2))
ax.bar(['CML', 'Bayesian'], [rmse_cml, rmse_bay], color=['steelblue','darkorange'])
ax.set_ylabel('RMSE (logit)')
ax.set_title('Person θ RMSE')
for k, v in enumerate([rmse_cml, rmse_bay]):
    ax.text(k, v + 0.01, f'{v:.3f}', ha='center', fontsize=11)

plt.tight_layout()
plt.suptitle('Person Parameter Recovery', y=1.01, fontsize=13)
plt.show()
print(f'Person θ: CML r={corr_cml:.3f}, RMSE={rmse_cml:.3f}')
print(f'Person θ: Bayes r={corr_bay:.3f}, RMSE={rmse_bay:.3f}')

In [ ]:
# ── 8. Item Parameter 비교: True vs CML vs Bayesian ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

lim_b = max(abs(b_true).max(), abs(b_cml).max(), abs(b_bayes).max()) + 0.3

# (a) True vs CML
ax = axes[0]
ax.scatter(b_true, b_cml, s=60, color='steelblue', zorder=3)
ax.plot([-lim_b, lim_b], [-lim_b, lim_b], 'r--')
ax.set_xlabel('True b (logit)'); ax.set_ylabel('CML b (logit)')
ax.set_title('Item b: True vs CML')
r_cml_b = np.corrcoef(b_true, b_cml)[0,1]
ax.text(0.05, 0.92, f'r = {r_cml_b:.3f}', transform=ax.transAxes, fontsize=10)

# (b) True vs Bayesian (with CI)
ax = axes[1]
ax.errorbar(b_true, b_bayes, yerr=1.96*b_bayes_sd,
            fmt='s', alpha=0.7, capsize=3, markersize=6, color='darkorange')
ax.plot([-lim_b, lim_b], [-lim_b, lim_b], 'r--')
ax.set_xlabel('True b (logit)'); ax.set_ylabel('Posterior mean b')
ax.set_title('Item b: True vs Bayesian (±1.96 SD)')
r_bay_b = np.corrcoef(b_true, b_bayes)[0,1]
ax.text(0.05, 0.92, f'r = {r_bay_b:.3f}', transform=ax.transAxes, fontsize=10)

# (c) CML vs Bayesian item estimates
ax = axes[2]
ax.scatter(b_cml, b_bayes, s=60, color='purple', zorder=3)
ax.plot([-lim_b, lim_b], [-lim_b, lim_b], 'r--')
ax.set_xlabel('CML b'); ax.set_ylabel('Bayesian b')
ax.set_title('Item b: CML vs Bayesian')
r_cb = np.corrcoef(b_cml, b_bayes)[0,1]
ax.text(0.05, 0.92, f'r = {r_cb:.3f}', transform=ax.transAxes, fontsize=10)

plt.tight_layout()
plt.suptitle('Item Parameter Recovery', y=1.01, fontsize=13)
plt.show()

rmse_b_cml = np.sqrt(np.mean((b_cml - b_true)**2))
rmse_b_bay = np.sqrt(np.mean((b_bayes - b_true)**2))
print(f'Item b: CML  r={r_cml_b:.3f}, RMSE={rmse_b_cml:.3f}')
print(f'Item b: Bayes r={r_bay_b:.3f}, RMSE={rmse_b_bay:.3f}')

# Posterior infit CI plot
fig2, ax2 = plt.subplots(figsize=(10, 3.5))
x = np.arange(I)
ax2.bar(x, infit_bayes_mean, color='steelblue', alpha=0.6, label='Posterior Infit mean')
ax2.errorbar(x, infit_bayes_mean, yerr=[infit_bayes_mean - infit_bayes_lo, infit_bayes_hi - infit_bayes_mean],
             fmt='none', capsize=4, color='navy', linewidth=1.5, label='95% CI')
ax2.axhline(1.3, color='red', linestyle='--', linewidth=1, label='Upper limit 1.3')
ax2.axhline(0.7, color='red', linestyle='--', linewidth=1, label='Lower limit 0.7')
ax2.axhline(1.0, color='gray', linestyle=':', linewidth=1)
ax2.set_xticks(x); ax2.set_xticklabels([f'I{i+1}' for i in x], fontsize=8)
ax2.set_ylabel('Infit MNSQ'); ax2.set_title('Posterior Infit Distribution (mean + 95% CI)')
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 9. 소규모 클래스(≤ 50명)에서의 RMM 적용 가능성

### 이론적 근거

CML은 **피험자 수 J가 고정된 상태에서 I(문항 수)가 증가**할 때 일치 추정량을 보장합니다.  
J=50, I=20~30이면 J와 I가 모두 적당한 규모로, 다음 조건이 충족되면 RMM을 적용할 수 있습니다.

| 조건 | 권장 값 | 근거 |
|------|---------|------|
| 최소 피험자 수 | J ≥ 30 | Linacre (1994): J < 30이면 b 추정 불안정 |
| 최소 문항 수 | I ≥ 10 | 안정적 person fit을 위해 |
| 극단 점수 비율 | < 5% | 극단 점수(0 or I) 제거 후 |

### 실용적 권고

- **Bayesian (Stan) 우선 권장**: 소표본에서 사전분포가 정규화 역할 → 안정적 추정
- 반드시 **부트스트랩 또는 posterior CI**로 추정 불확실성 확인
- 결과 해석 시 '교수 목적'에 집중: 학생 능력의 *순위*와 *상대적 난이도*만 해석
- 문항 수 10개 미만이면 CTT 병행 권장 (문항 정보량 부족)

### 결론

> J ≥ 30, I ≥ 10 조건에서 **Stan Bayesian RMM은 소규모 교실에서도 유효하게 적용 가능**합니다.  
> J < 30이면 결과를 탐색적으로만 해석하고, CTT와 병행하십시오.


## 10. CTT 대비 RMM 권장 이유

Classical Test Theory (CTT)는 적용이 쉽지만 **표본 의존적**입니다.

| 비교 항목 | CTT | RMM (Rasch) |
|-----------|-----|-------------|
| 문항 난이도 | 표본 의존 (쉬운 집단 → 쉬운 문항) | 표본 불변 (logit 척도) |
| 능력 추정 | 원점수 (정규화 필요) | 등간 척도 (logit) |
| 측정 오차 | 집단 수준 (표준 오차) | 개인 수준 (SEM이 능력별 상이) |
| 문항 적합도 | 항목-총점 상관 | Infit/Outfit (모델 기반) |
| 비교 가능성 | 동형 검사에서만 가능 | 문항 공유 시 다른 시험 비교 가능 |
| 소표본 안정성 | 표준화 계수 불안정 | Bayesian 사전분포로 보완 가능 |

### 교실 맥락에서의 이점

1. **등간 척도**: 능력 차이를 의미 있는 수치로 표현 (0.5 logit 차이는 어디서나 동일)
2. **개별 SEM**: 어떤 학생이 더 불확실한지 파악 가능
3. **문항 은행**: 서로 다른 시험을 같은 척도로 연결 → 공정한 비교
4. **부적합 문항 탐지**: Infit > 1.3인 문항은 다른 능력을 측정하거나 학생이 추측함
5. **DIF 탐지**: 성별·집단 간 편파 문항 자동 탐지 가능

> 소규모 교실이라도 **형성평가·총괄평가의 일관된 척도 유지**가 필요하다면 RMM이 CTT보다 우월합니다.


## 11. RMM이 CTT보다 진정 필요한 경우

CTT로 *충분하지 않고* RMM이 *반드시 필요한* 상황:

### (A) 측정 불변성이 요구될 때
- 서로 **다른 시험 버전**의 점수를 비교해야 할 때 (예: 1학기 vs 2학기 형성평가)
- 문항 부분 공유(anchor items)를 통한 **공통 척도** 유지 필요
- 동일 학생의 **성장(growth) 측정**: 원점수 비교는 문항 난이도가 다르면 불공정

### (B) 개별 측정 오차가 중요할 때
- 학습 부진 학생 진단: 해당 능력 범위에서의 SEM이 작아야 의미 있는 피드백 가능
- 커트라인 결정(Pass/Fail): 커트라인 근방의 불확실성을 명확히 표현해야 할 때

### (C) 문항 개발·은행 관리
- 문항을 재사용하거나 **CAT(Computerized Adaptive Testing)** 구현 시
- 교과 목표별 문항의 상대적 난이도 체계를 구축할 때

### (D) 집단 간 공정성 검증
- **DIF 분석**으로 성별·학교급·배경에 따른 편파 문항 탐지 (CTT는 이를 지원하지 않음)

### (E) 다국면 채점 (에세이, 실기)
- 채점자 엄격성(rater severity)을 보정해야 공정한 능력 추정 가능 → Many-Facet Rasch 필수

---

### 요약 판단 기준

| 상황 | CTT | RMM |
|------|-----|-----|
| 단순 수업 내 점수 산출 | ✓ 충분 | 선택 |
| 성장 측정 / 종단 비교 | ✗ 불충분 | ✓ 필수 |
| 집단 간 공정성 검증 | ✗ 불충분 | ✓ 필수 |
| 문항 은행 / CAT | ✗ 불가 | ✓ 필수 |
| 에세이 채점자 보정 | ✗ 불가 | ✓ 필수 |
| 커트라인 근방 정밀 판단 | △ 가능하나 조잡 | ✓ 권장 |

> **결론**: 교실이 작더라도 위 상황 중 하나라도 해당하면 RMM이 필요합니다.  
> Stan Bayesian RMM은 소표본의 불확실성을 정직하게 표현하므로, *"측정의 엄밀성"*과 *"실용성"* 사이에서 최선의 균형을 제공합니다.
